In [1]:
import pandas as pd
import numpy as np

In [2]:
languages = pd.read_csv('languages.csv')

In [3]:
parameters = pd.read_csv('parameters.csv')

In [4]:
values = pd.read_csv('values.csv')

In [5]:
codes = pd.read_csv('codes.csv')

In [6]:
print("Languages shape:", languages.shape)
print("Parameters shape:", parameters.shape)
print("Values shape:", values.shape)
print("Codes shape:", codes.shape)

Languages shape: (2467, 13)
Parameters shape: (195, 12)
Values shape: (441663, 9)
Codes shape: (398, 4)


In [7]:
# Filter for only GB020, GB021, GB022 parameters
# Note: The assignment mentions GB021 but the example shows GB023, I'll use GB020, GB022, GB023 as shown in the example
filtered_values = values[values['Parameter_ID'].isin(['GB020', 'GB022', 'GB023'])].copy()

# Remove unnecessary columns
columns_to_remove = ['ID', 'Comment', 'Source', 'Source_comment', 'Coders']
filtered_values = filtered_values.drop(columns=columns_to_remove, errors='ignore')

print("Filtered values shape:", filtered_values.shape)
print("Columns after filtering:", filtered_values.columns.tolist())

Filtered values shape: (7149, 4)
Columns after filtering: ['Language_ID', 'Parameter_ID', 'Value', 'Code_ID']


In [8]:
# Merge Name and Macroarea from languages dataframe
values_with_lang = filtered_values.merge(
    languages[['ID', 'Name', 'Macroarea']], 
    left_on='Language_ID', 
    right_on='ID', 
    how='left'
)

print("After merging languages:", values_with_lang.shape)
print("Columns:", values_with_lang.columns.tolist())

After merging languages: (7149, 7)
Columns: ['Language_ID', 'Parameter_ID', 'Value', 'Code_ID', 'ID', 'Name', 'Macroarea']


In [9]:
# Merge descriptions from codes dataframe
values = values_with_lang.merge(
    codes[['ID', 'Description']], 
    left_on='Code_ID', 
    right_on='ID', 
    how='left'
)

print("Final merged dataframe shape:", values.shape)
print("Columns:", values.columns.tolist())

# Display last few rows to verify
print("\nLast 3 rows:")
print(values.tail(3))

Final merged dataframe shape: (7149, 9)
Columns: ['Language_ID', 'Parameter_ID', 'Value', 'Code_ID', 'ID_x', 'Name', 'Macroarea', 'ID_y', 'Description']

Last 3 rows:
     Language_ID Parameter_ID Value  Code_ID      ID_x  Name      Macroarea  \
7146    zuni1245        GB020     1  GB020-1  zuni1245  Zuni  North America   
7147    zuni1245        GB022     0  GB022-0  zuni1245  Zuni  North America   
7148    zuni1245        GB023     0  GB023-0  zuni1245  Zuni  North America   

         ID_y Description  
7146  GB020-1     present  
7147  GB022-0      absent  
7148  GB023-0      absent  


In [10]:
# Merge descriptions from codes dataframe
values = values_with_lang.merge(
    codes[['ID', 'Description']], 
    left_on='Code_ID', 
    right_on='ID', 
    how='left'
)

print("Final merged dataframe shape:", values.shape)
print("Columns:", values.columns.tolist())

# Display last few rows to verify
print("\nLast 3 rows:")
print(values.tail(3))

Final merged dataframe shape: (7149, 9)
Columns: ['Language_ID', 'Parameter_ID', 'Value', 'Code_ID', 'ID_x', 'Name', 'Macroarea', 'ID_y', 'Description']

Last 3 rows:
     Language_ID Parameter_ID Value  Code_ID      ID_x  Name      Macroarea  \
7146    zuni1245        GB020     1  GB020-1  zuni1245  Zuni  North America   
7147    zuni1245        GB022     0  GB022-0  zuni1245  Zuni  North America   
7148    zuni1245        GB023     0  GB023-0  zuni1245  Zuni  North America   

         ID_y Description  
7146  GB020-1     present  
7147  GB022-0      absent  
7148  GB023-0      absent  


In [11]:
# Remove duplicate/spurious columns after merging
# Keep: Language_ID, Parameter_ID, Value, Code_ID, Name, Macroarea, Description
columns_to_keep = ['Language_ID', 'Parameter_ID', 'Value', 'Code_ID', 'Name', 'Macroarea', 'Description']
values = values[columns_to_keep]

# Rename columns for clarity
values = values.rename(columns={
    'Name': 'Language',
    'Parameter_ID': 'Feature',
    'Description': 'Value'
})

# Keep only necessary columns
values = values[['Language', 'Macroarea', 'Feature', 'Value']]

print("After cleaning and renaming:")
print(values.head())

After cleaning and renaming:
  Language  Macroarea Feature Value    Value
0    Abadi  Papunesia   GB020     ?      NaN
1    Abadi  Papunesia   GB022     ?      NaN
2    Abadi  Papunesia   GB023     ?      NaN
3  Mungbam     Africa   GB020     1  present
4  Mungbam     Africa   GB022     0   absent


In [12]:
# Reorder columns (already in correct order: Language, Macroarea, Feature, Value)
# Sort by macroarea, then by language name
values = values.sort_values(['Macroarea', 'Language']).reset_index(drop=True)

print("After sorting:")
print(values.head(10))

After sorting:
  Language Macroarea Feature Value    Value
0      Abé    Africa   GB020     1  present
1      Abé    Africa   GB022     0   absent
2      Abé    Africa   GB023     1  present
3  Acheron    Africa   GB020     0   absent
4  Acheron    Africa   GB022     0   absent
5  Acheron    Africa   GB023     0   absent
6    Acoli    Africa   GB020     0   absent
7    Acoli    Africa   GB022     0   absent
8    Acoli    Africa   GB023     0   absent
9    Afade    Africa   GB020     ?      NaN


In [13]:
# Remove lines with incomplete data (NaN values)
values = values.dropna()

# Replace feature names with more mnemonic ones
feature_mapping = {
    'GB020': 'defArt',
    'GB022': 'prenom', 
    'GB023': 'postnom'
}

values['Feature'] = values['Feature'].replace(feature_mapping)

print("After cleaning and renaming features:")
print(values.tail(3))
print(f"\nTotal rows: {len(values)}")

After cleaning and renaming features:
     Language      Macroarea  Feature Value    Value
7139   Yámana  South America   defArt     0   absent
7140   Yámana  South America   prenom     0   absent
7141   Yámana  South America  postnom     1  present

Total rows: 6607


In [14]:
# Create hierarchical index
values_hierarchical = values.set_index(['Macroarea', 'Language'])

print("Hierarchical index created:")
print(values_hierarchical.head(10))
print(f"\nIndex levels: {values_hierarchical.index.names}")

Hierarchical index created:
                    Feature Value    Value
Macroarea Language                        
Africa    Abé        defArt     1  present
          Abé        prenom     0   absent
          Abé       postnom     1  present
          Acheron    defArt     0   absent
          Acheron    prenom     0   absent
          Acheron   postnom     0   absent
          Acoli      defArt     0   absent
          Acoli      prenom     0   absent
          Acoli     postnom     0   absent
          Afar       defArt     0   absent

Index levels: ['Macroarea', 'Language']


3b